[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfc/janus-core/blob/main/docs/source/tutorials/cli/anharmonic_phonons.ipynb)

# Anharmonic Phonons with TDEP

`janus md` can write input files for [TDEP](https://tdep-developers.github.io/tdep/) after NVT molecular dynamics. TDEP then fits force constants to the sampled structures, including the effects of anharmonicity at the simulated temperature.

**Note:** `janus-core` only writes the input files for TDEP. To run TDEP, it must be [installed separately](https://stfc.github.io/janus-core/user_guide/installation.html#tdep), with its programs on `PATH`. Otherwise, the TDEP steps below are skipped.

## Set up environment (optional)

These steps are required to run this tutorial with Google Colab. To do so, uncomment and run the cell below.

These instructions but may work for other systems too, but it is typically preferable to prepare a virtual environment separately before running this notebook if possible.

In [ ]:
# ! uv pip install janus-core[mace] --system # Install janus-core with MACE

To ensure you have the latest version of `janus-core` installed, compare the output of the following cell to the latest version available at https://pypi.org/project/janus-core/

In [ ]:
from janus_core import __version__

print(__version__)

## Prepare structures

We write the two-atom primitive cell of NaCl, used as TDEP's unit cell, and the 64-atom cubic cell, used as the MD structure and TDEP's supercell. extxyz files keep the orientation of the cell, unlike CIF.

In [ ]:
from ase.build import bulk
from ase.io import write

unit_cell = bulk("NaCl", "rocksalt", a=5.64)
supercell = bulk("NaCl", "rocksalt", a=5.64, cubic=True) * (2, 2, 2)

write("NaCl-primitive.extxyz", unit_cell)
write("NaCl-supercell.extxyz", supercell)

## Run MD and write the TDEP inputs

TDEP inputs are requested through `post_process_kwargs`. Frames are saved from step 500, after equilibration, and every saved frame must have statistics from the same step, so `traj_every` must be a multiple of `stats_every`:

In [ ]:
%%writefile config-nvt-tdep.yml

ensemble: nvt
struct: NaCl-supercell.extxyz
arch: mace_mp
model: small

temp: 300
steps: 1000

traj_start: 500
traj_every: 50
stats_every: 50

post_process_kwargs:
  tdep_compute: True
  tdep_unit_cell_file: NaCl-primitive.extxyz
  tdep_supercell_file: NaCl-supercell.extxyz
  tdep_output_dir: janus_results/tdep-inputs

tracker: False

These settings, and others required by TDEP, such as using an NVT ensemble, are checked before the MD starts:

In [ ]:
! janus md --config config-nvt-tdep.yml

After the MD, the TDEP input files are in `janus_results/tdep-inputs`:

In [ ]:
! ls janus_results/tdep-inputs

## Run TDEP

If TDEP is installed, we can fit the second-order force constants and compute the phonon dispersion, which is written to `outfile.dispersion_relations`. See the [Python tutorial](../python/anharmonic_phonons.ipynb) for how to plot it.

In [ ]:
%%bash
if ! command -v extract_forceconstants > /dev/null; then
    echo "TDEP was not found on PATH, so it was not run"
    exit 0
fi

cd janus_results/tdep-inputs
extract_forceconstants -rc2 5 > extract_forceconstants.log
cp outfile.forceconstant infile.forceconstant
phonon_dispersion_relations > phonon_dispersion_relations.log
ls outfile.dispersion_relations*